In [6]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sn
import math as mt
import statsmodels.api as sm
from scipy import stats as st
from skimage.io import imread, imsave
from matplotlib import pyplot as plt
np.random.seed(42)
alpha = 0.05
n = 25
p = 3


### a) Определение влияния возраста на содержание иммуноглобулина

In [7]:
gr_1 = np.array([83, 85])
gr_2 = np.array([84, 85, 85, 86, 86, 87])
gr_3 = np.array([86, 87, 87, 87, 88, 88, 88, 88, 88, 89, 90])
gr_4 = np.array([89, 90, 90, 91])
gr_5 = np.array([90, 92])

groups = [gr_1, gr_2, gr_3, gr_4, gr_5]
psi = np.repeat(np.eye(5, dtype=int), [len(g) for g in groups], axis=0)
F = psi.T @ psi
F_inv = np.linalg.inv(F)
Y = np.concatenate(groups)

beta = F_inv @ psi.T @ Y

print(f'y = {beta[0]}*x1 + {beta[1]}*x2 + {beta[2]}*x3 + {beta[3]}*x4 + {beta[4]}*x5 + e')

Y_pred = psi @ beta
e = Y - Y_pred
RSS = e.T @ e
TSS = np.sum((Y - np.mean(Y))**2)
R2 = 1 - RSS / TSS

print(f'R2 = {R2}')

print('\nПроверка значимости коэффициентов')
for i in range(len(beta)):
    delta = beta[i] * np.sqrt(n - p) / np.sqrt(F_inv[i, i] * RSS)
    p_value = 2 * (1 - st.t.cdf(abs(delta), n - p))
    if p_value < alpha:
        print(f'p_value = {p_value} Коэффициент beta_{i} значим')
    else:
        print(f'p_value = {p_value} Коэффициент beta_{i} не значим')

y = 84.0*x1 + 85.5*x2 + 87.81818181818181*x3 + 90.0*x4 + 91.0*x5 + e
R2 = 0.8106060606060606

Проверка значимости коэффициентов
p_value = 0.0 Коэффициент beta_0 значим
p_value = 0.0 Коэффициент beta_1 значим
p_value = 0.0 Коэффициент beta_2 значим
p_value = 0.0 Коэффициент beta_3 значим
p_value = 0.0 Коэффициент beta_4 значим


### b) Попарное сравнение средних

In [ ]:
m = len(beta) * (len(beta) - 1) // 2

pairs = []
for i in range(len(beta)):
    for j in range(i+1, len(beta)):
        delta = (beta[i] - beta[j]) * np.sqrt(n - p) / np.sqrt((F_inv[i, i] + F_inv[j, j]) * RSS)
        p_val = 2 * (1 - st.t.cdf(abs(delta), n - p))
        pairs.append((p_val, i+1, j+1))

pairs.sort(key=lambda x: x[0])

rejected = True
for k, (p_val, i, j) in enumerate(pairs, start=1):
    threshold = alpha / (k)
    if p_val >= threshold:
        rejected = False
    result = 'beta_{i} != beta_{j}' if rejected else 'beta_{i} == beta_{j}'
    print(f'[{k}] beta_{i} и beta_{j}: p_value = {p_val:.6f}, порог = {alpha}/{k} = {threshold:.6f} -> {result}')

[1] beta_1 и beta_5: p_value = 0.0000, порог = 0.05/1 = 0.0500 -> beta_{i} != beta_{j}
[2] beta_2 и beta_4: p_value = 0.0000, порог = 0.05/2 = 0.0250 -> beta_{i} != beta_{j}
[3] beta_1 и beta_4: p_value = 0.0000, порог = 0.05/3 = 0.0167 -> beta_{i} != beta_{j}
[4] beta_2 и beta_5: p_value = 0.0000, порог = 0.05/4 = 0.0125 -> beta_{i} != beta_{j}
[5] beta_1 и beta_3: p_value = 0.0001, порог = 0.05/5 = 0.0100 -> beta_{i} != beta_{j}
[6] beta_2 и beta_3: p_value = 0.0002, порог = 0.05/6 = 0.0083 -> beta_{i} != beta_{j}
[7] beta_3 и beta_5: p_value = 0.0006, порог = 0.05/7 = 0.0071 -> beta_{i} != beta_{j}
[8] beta_3 и beta_4: p_value = 0.0014, порог = 0.05/8 = 0.0063 -> beta_{i} != beta_{j}
[9] beta_1 и beta_2: p_value = 0.0870, порог = 0.05/9 = 0.0056 -> beta_{i} == beta_{j}
[10] beta_4 и beta_5: p_value = 0.2723, порог = 0.05/10 = 0.0050 -> beta_{i} == beta_{j}
